In [ ]:
# import json
import os

import dotenv
import requests
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import (  # FunctionTool,; RequiredFunctionToolCall,
    ListSortOrder,
    MessageRole,
    SubmitToolOutputsAction,
    ToolOutput,
)
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

BALANCE_AGENT_ID = os.environ["BALANCE_AGENT_ID"]
PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]
BALANCE_API_BASE_URL = os.environ.get("BALANCE_API_BASE_URL", "http://localhost:8000")

In [ ]:
client = AgentsClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

In [ ]:
# def call_agent(user_input: str) -> str:
#     thread = client.threads.create()
#     # print('thread id:', thread.id)

#     client.messages.create(
#         thread_id=thread.id,
#         role=MessageRole.USER,
#         content=user_input,
#     )

#     client.runs.create_and_process(
#         thread_id=thread.id,
#         agent_id=BALANCE_AGENT_ID,
#     )

#     messages = client.messages.list(
#         thread_id=thread.id,
#         order=ListSortOrder.ASCENDING,
#     )

#     return messages

In [ ]:
def get_current_balance(customer_id: str) -> str:
    base_url = BALANCE_API_BASE_URL.rstrip("/")
    url = f"{base_url}/api/balance"
    print("get_current_balance() called with:", customer_id, "->", url)

    response = requests.post(
        url,
        json={"customer_id": customer_id},
        timeout=5,
    )
    print("mock status_code:", response.status_code, "body:", response.text)
    response.raise_for_status()
    return response.text


def call_agent(user_input: str):
    client = AgentsClient(
        endpoint=PROJECT_ENDPOINT,
        credential=DefaultAzureCredential(),
    )

    with client:
        thread = client.threads.create()

        client.messages.create(
            thread_id=thread.id,
            role=MessageRole.USER,
            content=user_input,
        )

        run = client.runs.create(
            thread_id=thread.id,
            agent_id=BALANCE_AGENT_ID,
        )

        while True:
            run = client.runs.get(
                thread_id=thread.id,
                run_id=run.id,
            )
            print("RUN STATUS:", run.status)

            if run.status == "requires_action":
                action = run.required_action

                if isinstance(action, SubmitToolOutputsAction):
                    tool_calls = action.submit_tool_outputs.tool_calls
                else:
                    print(
                        "Required action no es SubmitToolOutputsAction:", type(action)
                    )
                    break

                tool_outputs: list[ToolOutput] = []

                for tool_call in tool_calls:
                    print("TOOL CALL:", tool_call, type(tool_call))
                    output = functions.execute(tool_call)
                    print("TOOL OUTPUT (raw):", output)
                    tool_outputs.append(
                        ToolOutput(
                            tool_call_id=tool_call.id,
                            output=output,
                        )
                    )

                run = client.runs.submit_tool_outputs(
                    thread_id=thread.id,
                    run_id=run.id,
                    tool_outputs=tool_outputs,
                )
                continue

            if run.status in ("completed", "failed", "cancelled"):
                break

        messages = client.messages.list(
            thread_id=thread.id,
            order=ListSortOrder.ASCENDING,
        )

        return list(messages)

In [ ]:
get_current_balance('customer-001')

In [ ]:
# messages = call_agent("Super califragilistico espialidoso")
messages = call_agent(
    "Hola. Mi customer_id es customer-002. ¿Cuál es el saldo actual de mi cuenta?"
)

for msg in messages:
    print(f"{msg.role}:")
    for text_msg in msg.text_messages:
        print(text_msg.text.value)
    print()